In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
!pip install Kmodes
from kmodes.kmodes import KModes
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances




#Read the Excel file
df = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx')
df2 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
df3 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Loyalty')

print(df.head())

In [ ]:
# Βασικός καθαρισμός A, B, C
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = (
        df2[col]
        .astype(str)
        .str.strip()
        .str.title()
    )
    df2[col] = df2[col].replace(["Nan", "None", "Na", ""], np.nan)

# 2️⃣ Δημιουργούμε νέα στήλη CustomCategory (ξεκινάει ίδια με Category B)
df2["CustomCategory"] = df2["Category B"].copy()

# =========================================================
# 3️⃣ ΟΛΕΣ ΟΙ ΠΑΛΙΕΣ ΑΛΛΑΓΕΣ ΣΟΥ ΠΑΝΩ ΣΤΗΝ CustomCategory
# =========================================================

# 3.1 Συσκευασμενο → "Category C + ' σε συσκευασία'"
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = (
    df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"
)

# 3.2 Merge γαλακτοκομικών σε ενιαία κατηγορία (θα το σπάσουμε μετά)
to_merge_dairy = [
    "Γιαουρτια σε συσκευασία",
    "Τυροκομικα σε συσκευασία",
    "Γαλατα σε συσκευασία",
    "Βουτυρα σε συσκευασία",
    "Κρεμα Γαλακτος σε συσκευασία",
]
df2["CustomCategory"] = df2["CustomCategory"].replace(
    to_merge_dairy, "Γαλακτοκομικά σε συσκευασία"
)

# 3.3 Ρουχων + Ενδυση → Ρούχα & Ενδυση
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση"
)

# 3.4 Μπυρες + Κρασια + Οινοπνευματωδη → Οινοπνευματωδη
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη"
)

# 3.5 Σωματος / Ξυριστικα / Χεριων / Προσωπου → Προϊόντα Προσωπικής Φροντίδας
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"],
    "Προϊόντα Προσωπικής Φροντίδας",
)

# 3.6 Βαμβακια / Πανες Ακρατειας → Προιοντα Χαρτου
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου"
)

# 3.7 Μωρομαντηλα / Πανες Παιδικες → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μωρομαντηλα", "Πανες Παιδικες"], "Παιδικα"
)

# 3.8 Βρεφικη Τροφη → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Βρεφικη Τροφη", "Παιδικα"
)

# 3.9 Χυμοι / Ροφηματα → Χυμοί & Ροφήματα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία"],
    "Χυμοί & Ροφήματα",
)
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Αναψυκτικα", "Χυμοί & Ροφήματα"
)

# 3.10 Κρεας σε συσκευασία → Κατεψυγμενα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Κρεας σε συσκευασία", "Κατεψυγμενα"
)

# 3.11 Σαλτσες / Dressings → Σάλτσες & Dressings
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σαλτσες", "Dressings"], "Σάλτσες & Dressings"
)

# =========================================================
# 4️⃣ ΟΛΕΣ ΟΙ ΝΕΕΣ "ΕΞΥΠΝΕΣ" ΑΛΛΑΓΕΣ ΑΠΟ ΤΗΝ ΑΝΑΛΥΣΗ
# =========================================================

# 4.1 Split Ρούχα & Ενδυση → Προϊόντα Πλυντηρίου Ρούχων vs Ρούχα
laundry_items = [
    "Υγρα Πλυντηριου",
    "Μαλακτικα Πλυντηριου",
    "Ενισχυτικα-Χρωμοπαγιδες",
    "Σκονη Πλυντηριου",
    "Ταμπλετες Πλυντηριου",
    "Αποσκληρυντικα Πλυντηριου",
    "Πλυσιμο Στο Χερι",
    "Σιδερωματος",
]

mask_laundry = (
    (df2["CustomCategory"] == "Ρούχα & Ενδυση")
& (df2["Category C"].isin(laundry_items))
)
df2.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"

# 4.2 Διάλυση "Χυμα" σε λογικές κατηγορίες
xuma_map = {
    "Τυροκομικα": "Γαλακτοκομικά σε συσκευασία",
    "Αλλαντικα": "Αλλαντικα σε συσκευασία",
    "Μαναβικη": "Μαναβικη σε συσκευασία",
    "Ξηροι Καρποι": "Αλμυρα Σνακ",
    "Χαλβας": "Χαλβαδες Ταχινι",
    "Αλιπαστα": "Κονσερβες",
    "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",
}
mask_xuma = df2["Category B"] == "Χυμα"
df2.loc[mask_xuma, "CustomCategory"] = (
    df2.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")
)

# 4.3 Αλευρι από Αρτοσκευασματα → Βασικά Υλικά Μαγειρικής
mask_alevri = (df2["Category B"] == "Αρτοσκευασματα") & (df2["Category C"] == "Αλευρι")
df2.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"

# 4.4 Σπάσιμο Γαλακτοκομικών σε επιμέρους κατηγορίες
dairy_split_map = {
    "Γιαουρτια": "Γιαουρτια",
    "Τυροκομικα": "Τυροκομικα",
    "Γαλατα": "Γαλατα",
    "Βουτυρα": "Βουτυρα",
    "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",
}
mask_dairy = df2["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
df2.loc[mask_dairy, "CustomCategory"] = (
    df2.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία")
)

# 4.5 Split Γλυκα Σνακ
mask_glyka = df2["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = df2["Category C"]

df2.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
df2.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
df2.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
df2.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
df2.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
df2.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"

# 4.6 Split Πρωινο
mask_proino = df2["CustomCategory"] == "Πρωινο"
c_pro = df2["Category C"]

# Ροφήματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]),
         "CustomCategory"] = "Ροφηματα Πρωινου"

# Δημητριακά
df2.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"

# Αλείμματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]),
         "CustomCategory"] = "Αλειμματα Πρωινου"

# Εβαπορε → Γαλατα
df2.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"

# 4.7 Split Χυμοί & Ροφήματα
mask_drinks = df2["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = df2["Category C"]

# Αναψυκτικά
df2.loc[mask_drinks & c_dr.isin(
    ["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]
), "CustomCategory"] = "Αναψυκτικα"

# Χυμοί & Νέκταρ
df2.loc[mask_drinks & c_dr.isin(
    ["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]
), "CustomCategory"] = "Χυμοι & Νεκταρ"

# Έτοιμα ροφήματα (Ice Tea, Ice Coffee κ.λπ.)
df2.loc[mask_drinks & c_dr.isin(
    ["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]
), "CustomCategory"] = "Rtd Ροφηματα"

# Ενεργειακά
df2.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"

# 4.8 Split Κατεψυγμενα
mask_frozen = df2["CustomCategory"] == "Κατεψυγμενα"
c_fr = df2["Category C"]

df2.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
df2.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
df2.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
df2.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
df2.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"

# 4.9 Split Αλμυρα Σνακ → Ξηροι Καρποι ξεχωριστά
mask_salty = df2["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = df2["Category C"]

df2.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"

# =========================================================
# 5️⃣ Κανόνας: μικρές κατηγορίες (<10 barcodes) → "Διαφορα"
# =========================================================
counts = df2["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
df2["CustomCategory"] = df2["CustomCategory"].replace(small_cats, "Διαφορα")

# Ζυμες Ψυγειου σε συσκευασία → Αρτοσκευασματα
df2.loc[df2["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"

# =========================================================
# 6️⃣ Γρήγορος έλεγχος
# =========================================================
print("Μοναδικές Category B       :", df2["Category B"].nunique())
print("Μοναδικές CustomCategory   :", df2["CustomCategory"].nunique())
print("\nTop 40 CustomCategory:")
print(df2["CustomCategory"].value_counts().head(40))

# Νέα ενότητα

In [ ]:
#exclude quantities < 1
print(df.shape)
df = df[df['Quantity'] >= 1]
print(df.shape)

In [ ]:
display(df.describe())

In [ ]:
#exclude non positive values
df = df[df['Value_'] > 0]
print(df.shape)

In [ ]:
#exclude non integer quantities
df = df[df['Quantity'] % 1 == 0]
print(df.shape)

In [ ]:
#create new column Price
df['Price'] = df['Value_'] / df['Quantity']
print(df.shape)

In [ ]:
#Remove baskets with no LoyaltyCard attached
df = df[df['LoyaltyCard_ID'].notna()]
print(df.shape)


In [ ]:
df = df[df['Value_'].notna()]
df = df[df['Barcode'].notna()]
df = df[df['Date_'].notna()]
df = df[df['Basket_ID'].notna()]
df = df[df['Quantity'].notna()]
print(df.shape)
df.head()

In [ ]:
#Exclude barcodes that are not contained in the list of real barcodes
df = df[df['Barcode'].isin(df2['Barcode'])]
print(df.shape)

In [ ]:
#Exclude transactions that did not contain cardid's where cardholder was known or his Status was na
# try to fix the na into NaN?
valid_cards = df3.loc[df3['Status'].str.contains('na', na=False), 'Cardholder']

df = df[~df['LoyaltyCard_ID'].isin(valid_cards)]
print(df.shape)

In [ ]:
#Convert date from string to datetime
df['Date_'] = pd.to_datetime(df['Date_'], errors='coerce', dayfirst=True)
df.head()

In [ ]:
df.replace(['na'], pd.NA, inplace=True)
df.describe()


In [ ]:
plt.hist(df['Quantity'], bins=20, edgecolor='black')
plt.xlabel('Quantity')
plt.ylabel('Frequency')
plt.title('Distribution of Quantity')
plt.show()

In [ ]:
# Do we delete?
#Q1 = df['Quantity'].quantile(0.25)
#Q3 = df['Quantity'].quantile(0.75)
#IQR = Q3 - Q1

#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR

# Filter the DataFrame
#df = df[(df['Quantity'] >= lower_bound) & (df['Quantity'] <= upper_bound)]
#print(df.shape)

In [ ]:
plt.hist(df['Quantity'], bins=20, edgecolor='black')
plt.xlabel('Quantity')
plt.ylabel('Frequency')
plt.title('Distribution of Quantity')
plt.show()

In [ ]:
plt.hist(df['Value_'], bins=20, edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution of Value')
plt.show()

In [ ]:
Q1 = df['Value_'].quantile(0.25)
Q3 = df['Value_'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter the DataFrame
df = df[(df['Value_'] >= lower_bound) & (df['Value_'] <= upper_bound)]
print(df.shape)

In [ ]:
plt.hist(df['Value_'], bins=20, edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution of Value')
plt.show()

In [ ]:
df.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
df = pd.merge(df, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')
df.head()

In [ ]:
#df2['Custom_Category'] = (
    #df2['Category C']
    #.combine_first(df2['Category B'])
    #.combine_first(df2['Category A'])
#)

In [ ]:
#df2.head()
#df2['Custom_Category'].nunique()
#df2['Category C'].nunique()
#df_counts = df2['Category C'].value_counts().reset_index()
#df_counts.columns = ['Category_C', 'Count']
#print(df_counts)

In [ ]:
#df2 = pd.merge(df2, df_counts, left_on='Custom_Category', right_on='Category_C', how='inner')
#df2.head()

In [ ]:
#df2['Custom_Category'] = np.where(df2['Count'] > 60, df2['Category C'], df2['Category B'])
#df2.head()
#df2['Custom_Category'].nunique()

In [ ]:
df = pd.merge(df, df2, left_on='Barcode', right_on='Barcode', how='inner')
df.head()

In [ ]:
df.drop(columns=['Category A', 'Category B', 'Category C','Category C'], inplace=True)
df.head()

In [ ]:
#df.to_excel("/content/drive/MyDrive/output.xlsx", index=False)

In [ ]:
#Create One-hot encoding for the data to be ready for clustering
basket_matrix = pd.crosstab(df['Basket_ID'], df['CustomCategory'])
basket_matrix = (basket_matrix > 0).astype(int)
print(basket_matrix)

In [ ]:
basket_matrix['Total_categories'] = basket_matrix.sum(axis=1)
basket_matrix.head()

In [ ]:
basket_matrix['Total_categories'].describe()

In [ ]:
# Filter out the 1 item baskets
basket_matrix = basket_matrix[(basket_matrix['Total_categories'] > 1)]
print(basket_matrix.shape)

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

kmeans = KMeans(random_state=0)
visualizer = KElbowVisualizer(kmeans, k=(1,11))

visualizer.fit(basket_matrix.drop(columns=['Total_categories']))
_ = visualizer.show()

In [ ]:
data = basket_matrix.drop(columns=['Total_categories'])

# Calculate costs for different k values
costs = []
K_range = range(2, 7)  # 1 to 10 clusters

for k in K_range:
    # For k=1, we can use a simple approach or skip since one cluster is trivial
    if k == 1:
        # For single cluster, cost is total dissimilarity from mode
        kmodes = KModes(n_clusters=1, init='Huang', n_init=5, random_state=0)
        kmodes.fit(data)
        costs.append(kmodes.cost_)
    else:
        kmodes = KModes(n_clusters=k, init='Huang', n_init=5, random_state=0)
        kmodes.fit(data)
        costs.append(kmodes.cost_)

# Plot the elbow curve
plt.figure(figsize=(8, 6))
plt.plot(K_range, costs, 'bx-', linewidth=2, markersize=8)
plt.xlabel('Number of clusters (k)', fontsize=12)
plt.ylabel('Cost (within-cluster dissimilarity)', fontsize=12)
plt.title('Elbow Method for k-modes', fontsize=14)
plt.xticks(K_range)
plt.grid(True, alpha=0.3)

# Mark the elbow point (you can do this visually or calculate it)
plt.show()

In [ ]:
#from yellowbrick.cluster import SilhouetteVisualizer

#plt.figure(figsize=(2 * 5,  10 * 4))

#scores = {}
#for n_clusters in range(2, 20):
    #plt.subplot(10, 2, n_clusters - 1)
    #kmeans = KMeans(n_clusters, random_state=42)
    #visualizer = SilhouetteVisualizer(kmeans, colors='yellowbrick')
    #visualizer.fit(basket_matrix.drop(columns=['Total_categories']))
    #scores[n_clusters] = visualizer.silhouette_score_
    #plt.title(f'clusters: {n_clusters} score: {visualizer.silhouette_score_}')

In [ ]:
#sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

In [ ]:
kmeans = KModes(n_clusters=4, n_init=10, random_state=42)
kmeans.fit(basket_matrix.drop(columns=['Total_categories']))

In [ ]:
#dist = pairwise_distances(basket_matrix.drop(columns=['Total_categories']).astype(bool).values, metric='jaccard')
#db = DBSCAN(eps=0.3, min_samples=5, metric='precomputed')
#basket_matrix['cluster'] = db.fit_predict(dist)

In [ ]:
basket_matrix['cluster'] = kmeans.labels_
print(basket_matrix['cluster'])

In [ ]:
basket_matrix['cluster'].value_counts()

In [ ]:
cluster_profiles = basket_matrix.groupby('cluster').mean().drop(columns=['cluster','Total_categories'], errors='ignore')

In [ ]:
cluster_profiles_no_dairy = cluster_profiles.drop(columns=['Τυροκομικα'], errors='ignore')

filtered_profiles_dict = {
    cluster: row[row >= 0.10].sort_values(ascending=False)
    for cluster, row in cluster_profiles_no_dairy.iterrows()
}

for cluster, series in filtered_profiles_dict.items():
    if not series.empty: # Only plot if there are categories meeting the threshold
        plt.figure(figsize=(8,4))
        series.plot(kind='bar')
        plt.title(f"Cluster {cluster} (No Dairy): Product Categories \u226510% Presence")
        plt.ylabel("Percentage of Baskets")
        plt.ylim(0, 1)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No categories met the >=10% presence threshold for Cluster {cluster} (No Dairy).")

1. Φόρτωση Δεδομένων & Καθαρισμός Ιεραρχίας

Φορτώνουμε τα απαραίτητα αρχεία (Hierachy Categories & Barcodes και POS Data). Επαναλαμβάνουμε τον καθαρισμό και όλες τις προσαρμογές που έγιναν στην CustomCategory (βήματα 1.x έως 5.x) από την αρχική ανάλυση. Αυτό είναι κρίσιμο, καθώς οι κατηγορίες αυτές θα χρησιμοποιηθούν ξανά για την εξαγωγή των χαρακτηριστικών καλαθιού.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Ορισμός διαδρομής αρχείου (υποτίθεται η ίδια που χρησιμοποιήθηκε αρχικά)
file_path = "POS_DATA_BAPR_2024-2025_updated (3).xlsx"

# 1. Φόρτωση ORIGINAL hierarchy
hier = pd.read_excel(file_path, sheet_name="Hierachy Categories & Barcodes")

# Βασικός καθαρισμός A, B, C
for col in ["Category A", "Category B", "Category C"]:
    hier[col] = (hier[col].astype(str).str.strip().str.title())
    hier[col] = hier[col].replace(["Nan", "None", "Na", ""], np.nan)

# Δημιουργούμε νέα στήλη CustomCategory (ξεκινάει ίδια με Category B)
hier["CustomCategory"] = hier["Category B"].copy()

# ΕΦΑΡΜΟΓΗ ΟΛΩΝ ΤΩΝ ΜΕΤΑΤΡΟΠΩΝ (3.x & 4.x)
# (Για συντομία, εφαρμόζονται όλες οι αλλαγές CustomCategory του αρχικού κώδικα)
mask_sysk = hier["CustomCategory"] == "Συσκευασμενο"
hier.loc[mask_sysk, "CustomCategory"] = (hier.loc[mask_sysk, "Category C"] + " σε συσκευασία")
to_merge_dairy = ["Γιαουρτια σε συσκευασία", "Τυροκομικα σε συσκευασία", "Γαλατα σε συσκευασία", "Βουτυρα σε συσκευασία", "Κρεμα Γαλακτος σε συσκευασία"]
hier["CustomCategory"] = hier["CustomCategory"].replace(to_merge_dairy, "Γαλακτοκομικά σε συσκευασία")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"], "Προϊόντα Προσωπικής Φροντίδας")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Μωρομαντηλα", "Πανες Παιδικες", "Βρεφικη Τροφη"], "Παιδικα")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία", "Αναψυκτικα"], "Χυμοί & Ροφήματα")
hier["CustomCategory"] = hier["CustomCategory"].replace("Κρεας σε συσκευασία", "Κατεψυγμενα")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Σαλτσες", "Dressings"], "Σάλτσες & Dressings")
laundry_items = ["Υγρα Πλυντηριου", "Μαλακτικα Πλυντηριου", "Ενισχυτικα-Χρωμοπαγιδες", "Σκονη Πλυντηριου", "Ταμπλετες Πλυντηριου", "Αποσκληρυντικα Πλυντηριου", "Πλυσιμο Στο Χερι", "Σιδερωματος"]
mask_laundry = ((hier["CustomCategory"] == "Ρούχα & Ενδυση") & (hier["Category C"].isin(laundry_items)))
hier.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"
xuma_map = {"Τυροκομικα": "Γαλακτοκομικά σε συσκευασία", "Αλλαντικα": "Αλλαντικα σε συσκευασία", "Μαναβικη": "Μαναβικη σε συσκευασία", "Ξηροι Καρποι": "Αλμυρα Σνακ", "Χαλβας": "Χαλβαδες Ταχινι", "Αλιπαστα": "Κονσερβες", "Βουτυρα": "Γαλακτοκομικά σε συσκευασία"}
mask_xuma = hier["Category B"] == "Χυμα"
hier.loc[mask_xuma, "CustomCategory"] = (hier.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα"))
mask_alevri = (hier["Category B"] == "Αρτοσκευασματα") & (hier["Category C"] == "Αλευρι")
hier.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"
dairy_split_map = {"Γιαουρτια": "Γιαουρτια", "Τυροκομικα": "Τυροκομικα", "Γαλατα": "Γαλατα", "Βουτυρα": "Βουτυρα", "Κρεμα Γαλακτος": "Κρεμα Γαλακτος"}
mask_dairy = hier["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
hier.loc[mask_dairy, "CustomCategory"] = (hier.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία"))
mask_glyka = hier["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = hier["Category C"]
hier.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
hier.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
hier.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
hier.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
hier.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
hier.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"
mask_proino = hier["CustomCategory"] == "Πρωινο"
c_pro = hier["Category C"]
hier.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]), "CustomCategory"] = "Ροφηματα Πρωινου"
hier.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"
hier.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]), "CustomCategory"] = "Αλειμματα Πρωινου"
hier.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"
mask_drinks = hier["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = hier["Category C"]
hier.loc[mask_drinks & c_dr.isin(["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]), "CustomCategory"] = "Αναψυκτικα"
hier.loc[mask_drinks & c_dr.isin(["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]), "CustomCategory"] = "Χυμοι & Νεκταρ"
hier.loc[mask_drinks & c_dr.isin(["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]), "CustomCategory"] = "Rtd Ροφηματα"
hier.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"
mask_frozen = hier["CustomCategory"] == "Κατεψυγμενα"
c_fr = hier["Category C"]
hier.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
hier.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
hier.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
hier.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
hier.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"
mask_salty = hier["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = hier["Category C"]
hier.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"
counts = hier["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
hier["CustomCategory"] = hier["CustomCategory"].replace(small_cats, "Διαφορα")
hier.loc[hier["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"

2. Επανα-υπολογισμός Basket Segmentation

Επειδή η τμηματοποίηση πελατών χρησιμοποιεί τη συμμετοχή των Basket Clusters ως χαρακτηριστικό, πρέπει να τρέξουμε ξανά την ανάλυση καλαθιού για να πάρουμε τις ετικέτες (Cluster 0, 1, 2) για κάθε καλάθι.

Εξάγουμε αριθμητικά χαρακτηριστικά (Total_Value, Unique_Items).

Εξάγουμε ποσοστιαία συμμετοχή κατηγοριών (Share_...).

Εφαρμόζουμε PCA για μείωση διαστάσεων στις στήλες ποσοστών.

Εφαρμόζουμε K-Means με K=3 (βάσει της προηγούμενης ανάλυσης) για να ταξινομήσουμε κάθε Basket_ID.

In [ ]:
# Φόρτωση POS Data
df_pos = pd.read_excel(file_path, sheet_name="POS Data")
df_pos = df_pos.rename(columns={"Value_": "Value", "Date_": "Date", "LoyaltyCard_ID": "LoyaltyCard_ID"}, errors="ignore")

# Συγχώνευση με Custom Categories και καθαρισμός
df_pos = df_pos.merge(hier[["Barcode", "CustomCategory"]], on="Barcode", how="left")
df_pos = df_pos[(df_pos["Quantity"] > 0) & (df_pos["Value"] > 0)].copy()
df_pos = df_pos[df_pos["Quantity"] % 1 == 0]

# 1. Basket-level features (Aggregation)
basket_feats = df_pos.groupby("Basket_ID").agg(
    Total_Value=("Value", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Unique_Items=("Barcode", "nunique")
)

# 2. Category value shares per basket
cat_value = df_pos.groupby(["Basket_ID", "CustomCategory"])["Value"].sum().unstack(fill_value=0)
cat_share = cat_value.div(cat_value.sum(axis=1).replace(0, 1), axis=0)
cat_share.columns = [f"Share_{c}" for c in cat_share.columns]
final_basket_df = basket_feats.join(cat_share, how="left").fillna(0)
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

# 3. PCA and K-Means (K=3)
scaler = StandardScaler()
X_base = scaler.fit_transform(final_basket_df[["Total_Value","Total_Quantity","Unique_Items"]])
X_shares_scaled = scaler.fit_transform(final_basket_df[share_cols])
pca = PCA(n_components=0.80, random_state=0)
X_pca = pca.fit_transform(X_shares_scaled)
X = np.hstack([X_base, X_pca])

# Fit K-Means με K=3
k_final = 5
kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10).fit(X)
final_basket_df["Cluster"] = kmeans.labels_

# Συγχώνευση του Basket Cluster πίσω στο αρχικό df_pos
df_pos = df_pos.merge(
    final_basket_df[["Cluster"]],
    left_on="Basket_ID",
    right_index=True,
    how="left"
)

3. Εξαγωγή Χαρακτηριστικών Πελατών (RFM & Συμπεριφορικά)

Φιλτράρουμε για πελάτες με κάρτα Loyalty (LoyaltyCard_ID > 0). Υπολογίζουμε τις κλασικές μετρικές RFM-like, καθώς και τις Συμπεριφορικές Μετρικές που προκύπτουν από την ανάλυση καλαθιού: τα ποσοστά συμμετοχής των Basket Clusters (Shares) στα συνολικά καλάθια του κάθε πελάτη.

In [ ]:
# (Code from Cell 1 & 2 is assumed to have run to define df_pos and final_basket_df with 'Cluster')

# Φιλτράρισμα για έγκυρους πελάτες (TIP6: Loyalty Card holders)
df_cust = df_pos[df_pos["LoyaltyCard_ID"] > 0].copy()

# ... (Υπολογισμός Recency, Frequency, Avg_Basket_Value, Avg_Unique_Items) ...
df_cust['Date'] = pd.to_datetime(df_cust['Date'], format='%d/%m/%Y', errors='coerce')
df_cust = df_cust.dropna(subset=['Date'])
current_date = df_cust['Date'].max() + pd.Timedelta(days=1)
recency_df = df_cust.groupby('LoyaltyCard_ID')['Date'].max().reset_index()
recency_df['Recency'] = (current_date - recency_df['Date']).dt.days

basket_ids_per_cust = df_cust.groupby('LoyaltyCard_ID')['Basket_ID'].unique()

customer_monetary = df_cust.groupby('LoyaltyCard_ID').agg(
    Frequency=('Basket_ID', 'nunique'),
)
Avg_Basket_Value_Series = basket_ids_per_cust.apply(
    lambda x: final_basket_df.loc[x, 'Total_Value'].mean()
).rename('Avg_Basket_Value')
Avg_Unique_Items_Series = basket_ids_per_cust.apply(
    lambda x: final_basket_df.loc[x, 'Unique_Items'].mean()
).rename('Avg_Unique_Items_Per_Basket')

# **ΚΡΙΣΙΜΟ ΒΗΜΑ**: Υπολογισμός των Basket Cluster Shares (Χρήση του Basket Segmentation)
cluster_counts_cust = df_cust.groupby(['LoyaltyCard_ID', 'Cluster'])['Basket_ID'].nunique().unstack(fill_value=0)
cluster_shares_cust = cluster_counts_cust.div(cluster_counts_cust.sum(axis=1), axis=0)
cluster_shares_cust.columns = [f'Share_Cluster_{int(c)}' for c in sorted(df_pos["Cluster"].unique())]

# 4. Final Customer Feature Matrix
customer_df = recency_df.set_index('LoyaltyCard_ID')[['Recency']]
customer_df = customer_df.join(customer_monetary, how='inner')
customer_df = customer_df.join(Avg_Basket_Value_Series, how='inner')
customer_df = customer_df.join(Avg_Unique_Items_Series, how='inner')
customer_df = customer_df.join(cluster_shares_cust, how='inner') # <-- ΕΔΩ ΕΝΣΩΜΑΤΩΝΟΝΤΑΙ ΤΑ SHARES

print("Customer DataFrame Head (Features for Segmentation):")
print(customer_df.head())

4. Κανονικοποίηση (Scaling) & Επιλογή Βέλτιστου K
Κανονικοποιούμε όλα τα χαρακτηριστικά (Recency, Frequency, Monetary, Shares) με τον StandardScaler (ώστε να έχουν μέσο όρο 0 και τυπική απόκλιση 1) για να μην επηρεάζεται η ομαδοποίηση από τη διαφορά στην κλίμακα των τιμών.Τρέχουμε MiniBatchKMeans για $K=2$ έως $6$ και υπολογίζουμε το Inertia (Elbow) και το Silhouette Score για να επιλέξουμε τον βέλτιστο αριθμό clusters.

In [ ]:
# ====================================================================
# 4. Κανονικοποίηση (Scaling) & Επιλογή Βέλτιστου K
# ====================================================================

# 4.1 Προσθήκη Monetary στο customer_df (αν δεν υπάρχει)
if 'Monetary' not in customer_df.columns:
    customer_monetary_calc = df_pos.groupby('LoyaltyCard_ID')['Value'].sum().rename('Monetary')
    customer_df = customer_df.join(customer_monetary_calc, how='inner')


# 4.2 Ορισμός Features για Scaling (Πλέον συμπεριλαμβάνουμε το Monetary)
customer_cols = ["Recency", "Frequency", "Monetary", "Avg_Basket_Value", "Avg_Unique_Items_Per_Basket"]
share_cols = [c for c in customer_df.columns if c.startswith("Share_Cluster_")]
customer_cols.extend(share_cols)

X_cust = customer_df[customer_cols].values
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust)

# Find Optimal K for Customers (Elbow/Silhouette)
inertias_cust = []
sil_scores_cust = []
K_range_cust = range(3, 10) # K=2, 3, 4, 5, 6

print("\n" + "="*80)
print("--- K-SELECTION FOR CUSTOMER SEGMENTATION (K=2 to 6) ---")
print(f"{'k':<5} {'Inertia':<15} {'Silhouette Score':<20}")
print("-" * 40)

for k in K_range_cust:
    # Χρησιμοποιούμε MiniBatchKMeans για ταχύτητα
    mbk_cust = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3).fit(X_cust_scaled)
    inertias_cust.append(mbk_cust.inertia_)
    
    # Silhouette Score
    try:
        sil_cust = silhouette_score(X_cust_scaled, mbk_cust.labels_, sample_size=5000, random_state=42)
    except ValueError:
        sil_cust = silhouette_score(X_cust_scaled, mbk_cust.labels_)
        
    sil_scores_cust.append(sil_cust)
    print(f"{k:<5} {inertias_cust[-1]:<15.4f} {sil_scores_cust[-1]:<20.6f}")

# Επιλογή K=5
if max(sil_scores_cust) > 0 and len(sil_scores_cust) >= 4:
    best_idx = int(np.argmax(sil_scores_cust))
    k_final_cust = list(K_range_cust)[best_idx]
else:
    k_final_cust = 5 

print(f"\n✓ Selected K = {k_final_cust} for Final Segmentation.")

5. Τελική Ομαδοποίηση (K=3) & Ερμηνεία

Εφαρμόζουμε την τελική ομαδοποίηση. Η περίληψη περιλαμβάνει τις μετρικές Share_Cluster_0/1/2, οι οποίες είναι τα στοιχεία από το Basket Segmentation, και μας επιτρέπουν να ερμηνεύσουμε τη σύνθεση της αγοραστικής συμπεριφοράς κάθε segment πελατών.

In [ ]:
# ====================================================================
# 5. Τελική Ομαδοποίηση (K) & Ερμηνεία
# ====================================================================

# Fit K-Means
# (Υποθέτουμε ότι το k_final_cust έχει υπολογιστεί σωστά στο Block 4)
kmeans_cust = KMeans(n_clusters=k_final_cust, random_state=42, n_init=10).fit(X_cust_scaled)
customer_df["Customer_Cluster"] = kmeans_cust.labels_

# 5.1 Υπολογισμός Final Customer Summary (RFM & Mission Shares)
customer_summary = customer_df.groupby("Customer_Cluster").agg(
    Count=("Customer_Cluster", "size"),
    Recency_Days_Avg=("Recency", "mean"),
    Frequency_Baskets_Avg=("Frequency", "mean"),
    Monetary_Total_Avg=("Monetary", "sum"),
    Avg_Basket_Value_Avg=("Avg_Basket_Value", "mean"),
    Avg_Unique_Items_Avg=("Avg_Unique_Items_Per_Basket", "mean"),
    **{f'{col}_Avg': (col, 'mean') for col in share_cols} 
).round(2)

# 5.2 Δημιουργία Λογικής Ονοματοδοσίας (Heuristic Naming)
# (Υποθέτουμε ότι το basket_mission_names έχει δημιουργηθεί στο Block 3)
final_customer_segments = {}
for cluster in customer_summary.index:
    row = customer_summary.loc[cluster]
    
    mission_shares_row = row[[c for c in row.index if c.startswith('Share_Cluster_')]]
    top_mission_id = mission_shares_row.idxmax().replace('_Avg', '').replace('Share_Cluster_', '')
    
    try:
        # Χρησιμοποιούμε το λεξικό από το Block 3
        top_mission_name = basket_mission_names.get(int(top_mission_id), f"Mission_{top_mission_id}")
    except NameError:
        top_mission_name = f"Mission_{top_mission_id}"
    
    # ΛΟΓΙΚΗ ΟΝΟΜΑΤΟΔΟΣΙΑΣ ΠΕΛΑΤΩΝ
    if row['Monetary_Total_Avg'] > customer_summary['Monetary_Total_Avg'].quantile(0.85):
        name = "VIP / Heavy Spender"
    elif row['Frequency_Baskets_Avg'] > customer_summary['Frequency_Baskets_Avg'].quantile(0.8):
        name = f"Daily/Frequent Shopper ({top_mission_name})"
    elif row['Recency_Days_Avg'] > customer_summary['Recency_Days_Avg'].max() * 0.85:
        name = "At-Risk / Lapsed"
    elif row['Avg_Basket_Value_Avg'] > customer_summary['Avg_Basket_Value_Avg'].quantile(0.75):
        name = "Stock-Up (High-Value)"
    else:
        name = "Mainstream / Mid-Value"
        
    final_customer_segments[cluster] = name

# 5.3 Προσθήκη Ονόματος Cluster στο customer_df
customer_df["Cluster_Name"] = customer_df["Customer_Cluster"].map(final_customer_segments)

print("\n--- Τελική Περίληψη Customer Segments ---")
display(customer_summary.to_markdown(numalign="left", stralign="left"))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

# Υποθέτουμε ότι οι πίνακες final_customer_df, final_basket_df, customer_baskets
# και η λίστα share_cols έχουν δημιουργηθεί στα προηγούμενα βήματα (Q1 & Q2 Features).

# ====================================================================
# ΜΕΡΟΣ 1: ΤΕΛΙΚΟ CLUSTERING (k=5) & ΣΥΝΟΨΗ
# ====================================================================
final_customer_df = customer_df.copy()
customer_baskets = df_pos[df_pos["LoyaltyCard_ID"].isin(final_customer_df.index)][["Basket_ID", "LoyaltyCard_ID"]].drop_duplicates()
# Χαρακτηριστικά πελάτη (Χρησιμοποιήστε τα σωστά ονόματα για την κλιμάκωση)
customer_cols = ["Recency", "Frequency", "Avg_Basket_Value", "Avg_Unique_Items_Per_Basket", 
                 "Share_Cluster_0", "Share_Cluster_1", "Share_Cluster_2"] # Αν τα clusters του basket είναι 3
X_cust = final_customer_df[customer_cols]
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust)
k_final_cust = 5 # Επιλογή k=5

# Fit K-Means
kmeans_cust = KMeans(n_clusters=k_final_cust, random_state=42, n_init=10).fit(X_cust_scaled)
customer_df["Customer_Cluster"] = kmeans_cust.labels_

# 5.1 Summarize Clusters (Μέσες τιμές μη κανονικοποιημένων δεδομένων για ερμηνεία)
customer_summary = customer_df.groupby("Customer_Cluster").agg(
    Count=("Customer_Cluster", "size"),
    Recency_Days_Avg=("Recency", "mean"),
    Frequency_Baskets_Avg=("Frequency", "mean"),
    Monetary_Total_Avg=("Monetary", "sum"), 
    Avg_Basket_Value_Avg=("Avg_Basket_Value", "mean"),
    Avg_Unique_Items_Avg=("Avg_Unique_Items_Per_Basket", "mean"),
    **{f'{col}_Avg': (col, 'mean') for col in share_cols} # Basket Shares
).round(2)

# 5.2 Δημιουργία Λογικής Ονοματοδοσίας (Heuristic Naming)
final_customer_segments = {}
for cluster in customer_summary.index:
    row = customer_summary.loc[cluster]
    
    # 1. Βρες το Top Mission ID
    mission_shares_row = row[[c for c in row.index if c.startswith('Share_Cluster_')]]
    top_mission_id = mission_shares_row.idxmax().replace('_Avg', '').replace('Share_Cluster_', '')
    
    # 2. Βρες το Όνομα Mission
    # (Εδώ υποθέτουμε ότι το basket_mission_names έχει δημιουργηθεί σωστά στο Block 3)
    try:
        top_mission_name = basket_mission_names.get(int(top_mission_id), f"Mission_{top_mission_id}")
    except (NameError, ValueError):
        top_mission_name = f"Mission_{top_mission_id}"
    
    # 3. ΛΟΓΙΚΗ ΟΝΟΜΑΤΟΔΟΣΙΑΣ ΠΕΛΑΤΩΝ
    if row['Monetary_Total_Avg'] > customer_summary['Monetary_Total_Avg'].quantile(0.85):
        name = "VIP / Heavy Spender"
    elif row['Frequency_Baskets_Avg'] > customer_summary['Frequency_Baskets_Avg'].quantile(0.8):
        name = f"Daily/Frequent Shopper ({top_mission_name})"
    elif row['Recency_Days_Avg'] > customer_summary['Recency_Days_Avg'].max() * 0.85:
        name = "At-Risk / Lapsed"
    elif row['Avg_Basket_Value_Avg'] > customer_summary['Avg_Basket_Value_Avg'].quantile(0.75):
        name = "Stock-Up (High-Value)"
    else:
        name = "Mainstream / Mid-Value"
        
    final_customer_segments[cluster] = name

# 5.3 Προσθήκη Ονόματος Cluster στο customer_df (ΔΙΟΡΘΩΣΗ)
customer_df["Cluster_Name"] = customer_df["Customer_Cluster"].map(final_customer_segments)

print("\n--- Τελική Περίληψη Customer Segments ---")
display(customer_summary.to_markdown(numalign="left", stralign="left"))


# ====================================================================
# 6. Οπτικοποίηση Προϊοντικού Προφίλ ανά Cluster (ΜΕΡΟΣ 1)
# ====================================================================

VISUAL_THRESHOLD = 0.02 # Όριο 2%
share_cols_all = [c for c in final_basket_df.columns if c.startswith("Share_")]

for cl in customer_summary.index:
    
    cluster_customer_ids = customer_df[customer_df["Customer_Cluster"] == cl].index
    
    # Βρες τα Basket_ID αυτών των πελατών
    cluster_baskets_df = df_pos[df_pos["LoyaltyCard_ID"].isin(cluster_customer_ids)]
    basket_ids_in_cluster = cluster_baskets_df["Basket_ID"].unique()
    
    # Φιλτράρισμα των shares των καλαθιών
    cluster_basket_shares = final_basket_df.loc[final_basket_df.index.isin(basket_ids_in_cluster), share_cols_all]
    
    # Υπολόγισε το μέσο όρο των shares
    mean_shares = cluster_basket_shares.mean().sort_values(ascending=False)
    
    # Καθαρισμός και φιλτράρισμα (Εξαίρεση Τυροκομικών μόνο για οπτικοποίηση)
    mean_shares.index = mean_shares.index.str.replace('Share_', '')
    mean_shares = mean_shares.drop('Τυροκομικα', errors='ignore') 
    
    # Φιλτράρισμα για να δούμε μόνο τα σημαντικά shares
    filtered_shares = mean_shares[mean_shares >= VISUAL_THRESHOLD]
    
    if not filtered_shares.empty:
        # Δημιουργία γραφήματος
        plt.figure(figsize=(10, 5))
        filtered_shares.plot(kind='bar', color='darkgreen')
        plt.title(f"Customer Cluster {cl}: {customer_df.loc[customer_df['Customer_Cluster'] == cl, 'Cluster_Name'].iloc[0]} (Top Products)")
        plt.ylabel("Average Basket Share by Cluster")
        plt.xlabel("Product Category")
        plt.xticks(rotation=45, ha='right')
        plt.ylim(0, filtered_shares.max() * 1.1)
        plt.tight_layout()
        plt.show() # Αντί για savefig/close

    else:
        print(f"No categories met the >= {VISUAL_THRESHOLD*100:.0f}% share threshold for Customer Cluster {cl}.")

print("\nΟλοκλήρωση: Τελική ανάλυση Customer Segments ολοκληρώθηκε.")

# ====================================================================
# 7. ΟΠΤΙΚΟΠΟΙΗΣΗ HEATMAP (Mission Distribution ανά Customer Cluster)
# ====================================================================

# 7.1 Υπολογισμός μέσου Mission Share ανά Customer Cluster
cluster_centers_df = customer_summary.copy()

# 7.2 Προετοιμασία δεδομένων για heatmap
mission_features = [col for col in customer_df.columns if col.startswith('Share_Cluster_')]
heatmap_data = customer_summary[[c for c in customer_summary.columns if c.endswith('_Avg') and c.startswith('Share_Cluster_')]].T

# Μετονομασία Index/Columns
mission_names_map = {f'Share_Cluster_{i}_Avg': f'Basket_C{i}' for i in range(k_final_cust)}
heatmap_data = heatmap_data.rename(index=mission_names_map)
heatmap_data.columns = customer_df.groupby('Customer_Cluster')['Cluster_Name'].first().values

# Μετατροπή σε ποσοστό (Mission Share)
# (Δεν χρειάζεται div, καθώς το Share_Cluster_*_Avg είναι ήδη το ποσοστό)

plt.figure(figsize=(14, 7))
sns.heatmap(
    heatmap_data, 
    annot=True,
    fmt=".1%",
    cmap="YlGnBu",
    linewidths=.5,
    linecolor='white',
)

plt.title('Ποσοστιαία Κατανομή των Shopping Missions (Basket Types) ανά Customer Cluster', fontsize=14, fontweight='bold')
plt.xlabel('Customer Cluster', fontsize=11)
plt.ylabel('Mission (Basket Cluster Share)', fontsize=11)
plt.tight_layout()
plt.show()